#**Data Collection**

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
from datetime import date
import hashlib
import time

SCRAPE_DATE = date.today().isoformat()
HEADERS = {"User-Agent": "Mozilla/5.0"}

def clean_html(text):
    if not text or not isinstance(text, str):
        return ""
    soup = BeautifulSoup(text, "lxml")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def make_job_id(source, title, company, url):
    base = f"{source}_{title}_{company}_{url}"
    return hashlib.md5(base.encode()).hexdigest()


# -------------------------------
# REMOTIVE
# -------------------------------
def scrape_remotive():
    categories = [
        "software-dev", "data", "devops",
        "product", "design", "marketing", "finance",
        "Human resouorces", "writing", "Data analysis", "Education",
        "Medical", "Legal", "QA","All others"
    ]
    jobs = []

    for cat in categories:
        url = f"https://remotive.com/api/remote-jobs?category={cat}"
        data = requests.get(url, timeout=15).json()

        for job in data.get("jobs", []):
            jobs.append({
                "job_id": make_job_id("remotive", job["title"], job["company_name"], job["url"]),
                "title": clean_html(job["title"]),
                "company": clean_html(job["company_name"]),
                "location": clean_html(job["candidate_required_location"]),
                "job_type": job["job_type"],
                "category": job["category"],
                "tags": ", ".join(job["tags"]),
                "description": clean_html(job["description"]),
                "source": "Remotive",
                "url": job["url"],
                "date_posted": job["publication_date"],
                "scrape_date": SCRAPE_DATE
            })
        time.sleep(1)

    return pd.DataFrame(jobs)


# -------------------------------
# REMOTEOK
# -------------------------------
def scrape_remoteok():
    url = "https://remoteok.com/api"
    data = requests.get(url, headers=HEADERS, timeout=15).json()
    jobs = []

    for job in data:
        if "id" not in job:
            continue

        jobs.append({
            "job_id": make_job_id("remoteok", job.get("position"), job.get("company"), job.get("url")),
            "title": clean_html(job.get("position")),
            "company": clean_html(job.get("company")),
            "location": clean_html(job.get("location")),
            "job_type": job.get("employment_type"),
            "category": job.get("tags", [None])[0],
            "tags": ", ".join(job.get("tags", [])),
            "description": clean_html(job.get("description")),
            "source": "RemoteOK",
            "url": job.get("url"),
            "date_posted": job.get("date"),
            "scrape_date": SCRAPE_DATE
        })

    return pd.DataFrame(jobs)

# -------------------------------
# ARBEITNOW (JSON)
# -------------------------------
def scrape_arbeitnow():
    url = "https://www.arbeitnow.com/api/job-board-api"
    data = requests.get(url, timeout=15).json()
    jobs = []

    for job in data.get("data", []):
        jobs.append({
            "job_id": make_job_id("arbeitnow", job["title"], job["company_name"], job["url"]),
            "title": job["title"],
            "company": job["company_name"],
            "location": "Remote" if job["remote"] else job.get("location"),
            "job_type": None,
            "category": None,
            "tags": ", ".join(job.get("tags", [])),
            "description": clean_html(job.get("description")),
            "source": "Arbeitnow",
            "url": job["url"],
            "date_posted": job["created_at"],
            "scrape_date": SCRAPE_DATE
        })

    return pd.DataFrame(jobs)


# -------------------------------
# HIMALAYAS
# -------------------------------
def scrape_himalayas(max_jobs=100):
    jobs = []
    limit = 20  # Their hard-coded max limit per request

    # We loop using 'offset' to get more than just the first 20
    for offset in range(0, max_jobs, limit):
        url = f"https://himalayas.app/jobs/api?limit={limit}&offset={offset}"

        try:
            response = requests.get(url, timeout=15)
            if response.status_code != 200:
                break

            data = response.json()
            batch = data.get("jobs", [])

            if not batch:
                break # Stop if we hit an empty page

            for job in batch:
                categories = job.get("categories", [])
                jobs.append({
                    "job_id": make_job_id("himalayas", job.get("title"), job.get("companyName"), job.get("applicationLink")),
                    "title": clean_html(job.get("title")),
                    "company": clean_html(job.get("companyName")),
                    "location": job.get("locationRestriction", "Remote"),
                    "job_type": job.get("employmentType"),
                    "category": categories[0] if categories else None,
                    "tags": ", ".join(categories),
                    "description": clean_html(job.get("description")),
                    "source": "Himalayas",
                    "url": job.get("applicationLink"),
                    "date_posted": job.get("pubDate"),
                    "scrape_date": SCRAPE_DATE
                })

            # Brief sleep to be polite since we are making multiple calls
            time.sleep(1)

        except Exception as e:
            print(f"Error on Himalayas offset {offset}: {e}")
            break

    return pd.DataFrame(jobs)






In [ ]:
all_jobs = []
print("Scraping Remotive...")
df_remotive = scrape_remotive()
print(f"Remotive jobs: {len(df_remotive)}")
all_jobs.append(df_remotive)

print("\nScraping RemoteOK...")
df_remoteok = scrape_remoteok()
print(f"RemoteOK jobs: {len(df_remoteok)}")
all_jobs.append(df_remoteok)

# print("\nScraping We Work Remotely...")
# df_wwr = scrape_weworkremotely()
# print(f"WeWorkRemotely jobs: {len(df_wwr)}")
# all_jobs.append(df_wwr)

print("\nScraping Arbeitnow...")
df_arbeitnow = scrape_arbeitnow()
print(f"Arbeitnow jobs: {len(df_arbeitnow)}")
# all_jobs.append(df_arbeitnow)


print("\nScraping Himalayas...")
df_himalayas = scrape_himalayas()
print(f"Himalayas jobs: {len(df_himalayas)}")
all_jobs.append(df_himalayas)





Scraping Remotive...
Remotive jobs: 40

Scraping RemoteOK...
RemoteOK jobs: 100

Scraping Arbeitnow...
Arbeitnow jobs: 100

Scraping Himalayas...
Himalayas jobs: 100


CLEANING ARBEITNOW DATA

In [ ]:
cleaned_titles = [
    'Trainee Social Media Marketing und Personal Branding',
    'Performance Marketing Teamlead im E-Commerce',
    "Didn't find a suitable position? – Show Us What You’ve Got with an Unsolicited Application!",
    'Sachbearbeiter Baufinanzierungen / Darlehensmanagement',
    'Software Test Engineer',
    'Trainee LinkedIn Marketing und Personal Branding',
    'Projektmanager E-Commerce',
    'Junior Projektmanager E-Commerce',
    'ERP-Consultant',
    'Netzwerk- und Systemadministrator Linux',
    'HR Business Partner im Recruiting',
    'Technischer Systemplaner HLSK / Versorgungstechnik',
    'Head of Recruiting',
    'Machine Learning Engineer',
    'Praktikant',
    'Personalsachbearbeiterin Minijob',
    'Werkstudenten / Praxissemester - Inside Sales & Marketing',
    'Kita-Manager',
    'Vertriebsmitarbeiter',
    'Viral Short-Form Copywriter – Reels, TikToks & YouTube Shorts',
    'Werkstudenten / Praxissemester - Software-Consulting',
    'Systemadministrator Windows-Netze',
    'Personalwesen',
    'Creative Strategist für große E-Commerce Kunden',
    'Praktikum Management-Assistent | Schatten der Geschäftsführung',
    'Specialist HR Operations - Payroll Specialist',
    'E-Commerce Manager - Partner Management & Shop Operations',
    'Performance Marketing Manager',
    'Studentische Hilfskraft Marketing & Social Media, Schwerpunkt KI & Content',
    'Performance Video Ad Creator - Meta & TikTok',
    'IT Operations Freelancer',
    'Werkstudent IT Operations',
    'Managementassistenz * Controlling',
    'Senior Controller / Bilanzbuchhalter',
    'Product Sub Editor & Translator Korean',
    'B2B SaaS Sales & Business Development Manager',
    'Technischer Mitarbeiter',
    'Geschäftsführer / Quereinsteiger / Franchise-Partner / Franchise-Nehmer / Selbstständiger / Unternehmer für den Standort Hildesheim',
    'Softwareentwickler WebRTC und TypeScript Backend für Cloud-Plattform',
    'Senior Labour Relations Manager',
    'Partner Success Manager',
    'Praktikant im E-Commerce | Online-Marketing',
    'Chief Commercial Officer Defence & Space',
    'Senior Software Engineer',
    'Backend Softwareentwickler',
    'Frontend Softwareentwickler',
    'HR-Administration- Personalsachbearbeiter',
    'QA Gametester',
    'IT - Supportexpert:in Finanzbuchhaltungssoftware',
    'Senior Controller - Unternehmenssteuerung & Transformation',
    'Oracle Database Administrator',
    'Werkstudent Corporate Finance & Controlling – Fokus CAPEX, Cashflow & Reporting',
    'Expert Network Engineer',
    'ERP-Software Helpdesk Mitarbeiter',
    'HR Generalist',
    'Abteilungsleiter Finanzbuchhaltung / Rechnungswesen',
    'Werkstudent SEO / GEO / Content Marketing',
    'TikTok Livestream / Live Host',
    'Fachkraft für Arbeitssicherheit / EHS Specialist',
    'Expert Network Engineer IP',
    'Werkstudent People & Culture',
    'Social Media Creator & Host',
    'Logistikcontroller / Controller / Produktionscontroller mit BI-Tools & SAP',
    'Senior Business Operations Specialist',
    'IT-Systemadministrator',
    '(Senior) Platform Engineer e*star',
    'SAP Consultant',
    'Technical Account Manager - DACH',
    'Senior UX-/UI-Designer (Figma) - MES-Anwendungen',
    'Senior Graphic Designer',
    '(Junior) Consultant für den Bereich Tech',
    'Personalsachbearbeiter in Teilzeit',
    'Technical Product & Implementation Specialist',
    'Verfahrensmechaniker für Kunststoff- und Kautschuktechnik',
    'Sachbearbeiter Finanzbuchhaltung',
    'Software Engineering Team Lead - Reach Project',
    'Manager Accounting',
    'Content Strategist für Paid Ads',
    'Social Media Strategist',
    'Praktikum Influencer Marketing',
    'Auszubildende Kaufmann/-frau für Büromanagement ab dem 01.08.26',
    'CFO / Kaufmännischer Leiter für Automobilzulieferer',
    'Ausbilder für Fachinformatik',
    'Working Student Graphic Design',
    'KFZ - Werkstattleiter für Autohausgruppe',
    'Senior IT System-Engineer',
    'Sales Controller für große Autohaus-Retail-Gruppe in KÖLN - Hybrid',
    'Personalberater',
    'Recruiter in Teilzeit',
    'Mitarbeiter im Backoffice',
    'Mediengestalter Digital und Print',
    'Head of Product & Operations',
    'Serviceassistenz / Empfangsmitarbeiter im Autohaus',
    'SEO Copywriter - Vollzeit',
    '(Senior) Personalcontroller',
    'Junior CRM Manager',
    'Traumpraktikum mit Perspektive: Werde Fachinformatiker für Anwendungsentwicklung bei uns in Berlin!',
    'Trainee CRM Manager',
    'Consultant / Projektmanager Logistik Warehouse Management System'
]

def clean_job_title(title):
    title = re.sub(r'\s*[\(\/].*?[mwdfgnx].*?[\)]', '', title, flags=re.IGNORECASE)
    title = re.sub(r'[*:/]in\b', '', title)
    title = re.sub(r'\s*[\-\|]\s*(100%|remote|hybrid|homeoffice|Düsseldorf|Köln|Berlin|Iserlohn|Othmarschen|Hildesheim).*', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s+', ' ', title).strip()
    return title




df_arbeitnow['title'] = df_arbeitnow['title'].apply(clean_job_title)

In [ ]:
category_map = {
    "Marketing & Creative": [
        "marketing", "social media", "branding", "e-commerce", "copywriter",
        "reels", "tiktoks", "content", "creative", "seo", "graphic design",
        "ux", "ui", "mediengestalter", "crm", "influencer", "video ad"
    ],
    "IT & Software Engineering": [
        "software", "engineer", "developer", "backend", "frontend", "linux",
        "network", "administrator", "it - support", "helpdesk", "sap",
        "machine learning", "tech", "platform engineer", "webrtc", "typescript",
        "database", "gametester", "system-engineer", "fachinformatik"
    ],
    "HR & Recruiting": [
        "hr", "recruiting", "personal", "people & culture", "recruiter",
        "labour relations", "payroll", "ausbilder", "personalberater"
    ],
    "Finance & Controlling": [
        "controller", "controlling", "finance", "accounting", "buchhaltung",
        "darlehensmanagement", "cfo", "rechnungswesen", "audit", "payroll"
    ],
    "Sales & Business Development": [
        "sales", "vertrieb", "business development", "account manager",
        "commercial", "partner success"
    ],
    "Operations & Management": [
        "projektmanager", "project manager", "geschäftsführer", "operations",
        "backoffice", "sachbearbeiter", "managementassistenz", "leiter",
        "head of", "chief", "consultant", "erp", "logistik"
    ],
    "Student & Entry Level": [
        "trainee", "praktikant", "praktikum", "werkstudent", "working student",
        "studentische", "auszubildende", "junior"
    ]
}


# 1. Define the search function
def find_category(title):
    title = str(title).lower()
    for category, keywords in category_map.items():
        for word in keywords:
            if word.lower() in title:
                return category
    return "Other"


df_arbeitnow['category'] = df_arbeitnow['title'].apply(find_category)

In [ ]:
df_arbeitnow['category']

,category
0,Other
1,Other
2,Other
3,Marketing & Creative
4,IT & Software Engineering
...,...
95,Other
96,Student & Entry Level
97,Other
98,Marketing & Creative


Making final data frame

In [ ]:
all_jobs.append(df_arbeitnow)

In [ ]:
# Combine all dataframes
final_df = pd.concat(all_jobs, ignore_index=True)

In [ ]:
final_df.isnull().sum()

,0
job_id,0
title,0
company,0
location,0
job_type,200
category,0
tags,0
description,0
source,0
url,0


#**Data Cleaning**

In [ ]:
# Clean tags column
final_df["tags"] = (
    final_df["tags"]  # Fixed: was df
    .str.lower()
    .str.replace("c\\+\\+", "c++", regex=True)
    .str.replace("nodejs", "node.js", regex=True)
)


In [ ]:
final_df['tags']

,tags
0,"video, ai/ml, research, game design, data anal..."
1,"css, excel, frontend, git, html, illustrator, ..."
2,"api, backend, git, ios, security, swift, ui/ux..."
3,"crm, google sheets, financial services, inside..."
4,"api, backend, django, excel, python, react, ai..."
...,...
335,"remote, it"
336,system and network administration
337,system and network administration
338,"remote, marketing manager"


In [ ]:
final_df["tags_list"] = final_df["tags"].str.split(",")

In [ ]:
final_df["tags_list"] = final_df["tags_list"].apply(
    lambda x: [i.strip() for i in x] if isinstance(x, list) else []
)

In [ ]:
final_df.shape

(340, 13)

In [ ]:
final_df.head(1)

,job_id,title,company,location,category,tags,description,source,url,date_posted,scrape_date,tags_list
0,4cf90830d8d06e1969b33c1b50c448fc,AI Trainer,Anuttacon,Worldwide,AI / ML,"video, ai/ml, research, game design, data anal...",🎮 Our Culture: Our culture thrives on creativi...,Remotive,https://remotive.com/remote-jobs/ai-ml/ai-trai...,2026-02-12T21:01:01,2026-02-15,"[video, ai/ml, research, game design, data ana..."


In [ ]:
# drop job_type column
final_df.drop(columns=["job_type"], inplace=True)

KeyError: "['job_type'] not found in axis"

In [ ]:
final_df.isnull().sum()

,0
job_id,0
title,0
company,0
location,0
category,0
tags,0
description,0
source,0
url,0
date_posted,0


In [ ]:
import plotly.express as px


location_map = {
    "North America": ["USA", "United States", "US Remote", "Canada", "Mexico", "Texas", "NY", "Washington", "CA"],
    "Germany": ["Berlin", "Munich", "Hamburg", "Düsseldorf", "Cologne", "Frankfurt", "Aachen", "Hildesheim"],
    "Europe (Other)": ["France", "Netherlands", "UK", "Slovakia", "Europe"],
    "APAC": ["Singapore", "Malaysia", "Manila", "Philippines", "APAC", "Kuala Lumpur"],
    "LATAM": ["Brazil", "SÃ£o Paulo", "Latin America", "Londrina"],
    "Middle East": ["Dubai", "Israel"],
    "Global/Remote": ["Worldwide", "Remote", "Remoto"]
}

def assign_region(location):
    if not location or str(location).strip() == "":
        return "Not Specified"

    loc_lower = str(location).lower()

    for region, keywords in location_map.items():
        if any(kw.lower() in loc_lower for kw in keywords):
            return region

    return "Other"

final_df['region'] = final_df['location'].apply(assign_region)

In [ ]:
final_df.head(1)

,job_id,title,company,location,category,tags,description,source,url,date_posted,scrape_date,tags_list,region
0,4cf90830d8d06e1969b33c1b50c448fc,AI Trainer,Anuttacon,Worldwide,AI / ML,"video, ai/ml, research, game design, data anal...",🎮 Our Culture: Our culture thrives on creativi...,Remotive,https://remotive.com/remote-jobs/ai-ml/ai-trai...,2026-02-12T21:01:01,2026-02-15,"[video, ai/ml, research, game design, data ana...",Global/Remote


In [ ]:
final_df['category'].isnull().sum()

np.int64(0)

In [ ]:
data = final_df.copy()

In [ ]:
data['category'].isnull().sum()

np.int64(0)

In [ ]:
final_df['category'].fillna("Other / General", inplace = True)

/tmp/ipython-input-2747873716.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  final_df['category'].fillna("Other / General", inplace = True)


In [ ]:
final_df['category'].unique()

array(['Sales', 'Software Engineering', 'Other / General',
       'Content & Writing', 'AI / Machine Learning', 'Marketing',
       'Product Management', 'DevOps & IT Infrastructure',
       'Customer Success & Support', 'Management & Leadership',
       'Information Security', 'Administrative Support',
       'Training & Education', 'SaaS Strategy', 'Web Development',
       'Consulting', 'Creative & Design', 'Data Science & Analytics',
       'HR & Recruiting', 'Project Management', 'Blockchain & Crypto',
       'Specialized Engineering', 'Finance & Accounting',
       'Business Development', 'Entry Level', 'Operations'], dtype=object)

Cleaning text entities and giving them bullet names

In [ ]:
simple_mapping = {
    # --- WEB & FRONTEND ---
    'web': 'Web Development',
    'web3': 'Web Development',
    'web development3': 'Web Development',
    'Senior-PHP-Developer': 'Web Development',
    'developer': 'Web Development',
    'web-development': 'Web Development',

    # --- SOFTWARE ENGINEERING ---
    'Software Development': 'Software Engineering',
    'IT & Software Engineering': 'Software Engineering',
    'Software-Engineer-II': 'Software Engineering',
    'engineer': 'Software Engineering',
    'architect': 'Software Engineering',
    'software': 'Software Engineering',
    'Alto-Software-Group': 'Software Engineering',
    'technical': 'Software Engineering',
    'game': 'Software Engineering',

    # --- AI, ML & DATA SCIENCE ---
    'AI / ML': 'AI / Machine Learning',
    'Workday-AI-Strategy-Lead': 'AI / Machine Learning',
    'python': 'AI / Machine Learning',
    'analyst': 'Data Science & Analytics',
    'Senior-Operations-Research-Scientist': 'Data Science & Analytics',
    'Senior-Business-Information-Analyst': 'Data Science & Analytics',
    'Senior-Supply-Chain-Analyst': 'Data Science & Analytics',
    'Physical-Scientist': 'Data Science & Analytics',
    'Annotator': 'Data Science & Analytics',
    'Senior-Credit-Analyst': 'Data Science & Analytics',
    'Science': 'Data Science & Analytics',

    # --- SALES & REVENUE ---
    'Sales': 'Sales',
    'Sales / Business': 'Sales',
    'Sales-Specialist': 'Sales',
    'Inside-Sales-Representative': 'Sales',
    'Senior-Sales-Executive': 'Sales',
    'National-Account-Executive': 'Sales',
    'Sales-EMEA': 'Sales',
    'VP-Of-Financial-Institution-Sales': 'Sales',
    'Business-Development': 'Business Development',
    'Sales & Business Development': 'Business Development',
    'Healthcare-Business-Development-Director': 'Business Development',

    # --- MARKETING & CREATIVE ---
    'Marketing': 'Marketing',
    'Marketing & Creative': 'Marketing',
    'Associate-Marketing-Specialist': 'Marketing',
    'Head-Of-Influencer-Marketing': 'Marketing',
    'growth': 'Marketing',
    'design': 'Creative & Design',
    'Music': 'Creative & Design',

    # --- CONTENT & WRITING ---
    'Writing': 'Content & Writing',
    'writer': 'Content & Writing',
    'Content': 'Content & Writing',
    'Creator-(Writer)': 'Content & Writing',
    'Writing-Editing': 'Content & Writing',
    'Editorial-Assistant': 'Content & Writing',
    'Create': 'Content & Writing',

    # --- IT, DEVOPS & INFRASTRUCTURE ---
    'DevOps / Sysadmin': 'DevOps & IT Infrastructure',
    'DevOps-Engineer': 'DevOps & IT Infrastructure',
    'ansible': 'DevOps & IT Infrastructure',
    'system': 'DevOps & IT Infrastructure',
    'Information-Technology': 'DevOps & IT Infrastructure',
    'Operations-Engineer': 'DevOps & IT Infrastructure',
    'IT-Portfolio-Management-Analyst': 'DevOps & IT Infrastructure',

    # --- CUSTOMER SUCCESS & SUPPORT ---
    'support': 'Customer Success & Support',
    'Customer Service': 'Customer Success & Support',
    'Customer-Support-Specialist': 'Customer Success & Support',
    'Client-Services': 'Customer Success & Support',
    'French-Speaking-Customer-Success-Associate': 'Customer Success & Support',
    '&-Customer-Success': 'Customer Success & Support',
    'Product-Support-Specialist': 'Customer Success & Support',
    'Remote-Customer-Service-Associate': 'Customer Success & Support',
    'assistant': 'Administrative Support',

    # --- PRODUCT & PROJECT MANAGEMENT ---
    'Product': 'Product Management',
    'Enterprise-Product-Manager': 'Product Management',
    'Remote-Tax-Software-Product-Expert': 'Product Management',
    'coordinator': 'Project Management',
    'Service-Coordinator': 'Project Management',
    'Senior-Program-Director': 'Project Management',
    'Portfolio-Management-Director': 'Project Management',
    'Sr.-Professional-Services-Practice-Manager': 'Project Management',
    'Risk-Management-Program-Manager': 'Project Management',

    # --- HR, RECRUITING & TRAINING ---
    'hr': 'HR & Recruiting',
    'recruiter': 'HR & Recruiting',
    'HR & Recruiting': 'HR & Recruiting',
    'Career-Development-Manager': 'HR & Recruiting',
    'training': 'Training & Education',
    'Language-&-Linguistics': 'Training & Education',

    # --- FINANCE & LEGAL ---
    'Finance & Controlling': 'Finance & Accounting',
    'Accounting': 'Finance & Accounting',
    'Finnish-Payroll-Specialist': 'Payroll Services',
    'Norwegian-Payroll-Specialist': 'Payroll Services',
    'Remote-Senior-Insurance-Analyst': 'Finance & Accounting',
    'Claims-Investigation': 'Legal & Compliance',
    'Regulatory-Affairs-Specialist-II': 'Legal & Compliance',
    '&-Compliance': 'Legal & Compliance',

    # --- HEALTH & CLINICAL ---
    'Clinical': 'Healthcare & Clinical',
    'Hospital-&-Health-Care': 'Healthcare & Clinical',
    'Healthcare-Advisor': 'Healthcare & Clinical',
    'Behavioral-Health-Manager': 'Healthcare & Clinical',
    'Medical-Coding': 'Healthcare & Clinical',
    'Biomedical-Equipment-Technician': 'Healthcare & Clinical',

    # --- OPERATIONS & LEADERSHIP ---
    'Operations & Management': 'Operations',
    'Business-Operations': 'Operations',
    'Administrative-Services': 'Operations',
    'Outsourcing-Offshoring': 'Operations',
    'manager': 'Management & Leadership',
    'director': 'Management & Leadership',

    # --- SPECIALIZED ROLES ---
    'Engineering': 'Specialized Engineering',
    'Field-Service-Engineer': 'Specialized Engineering',
    'Construction': 'Specialized Engineering',
    'Utility-Forestry-Consultant': 'Specialized Engineering',
    'Welding-Safety-Manager': 'Safety & Environment',
    'Senior-EHS-(Environment': 'Safety & Environment',
    'Golf-Retail-Specialist': 'Retail & Hospitality',
    'crypto': 'Blockchain & Crypto',
    'saas': 'SaaS Strategy',
    'consultant': 'Consulting',
    'security': 'Information Security',
    'infosec': 'Information Security',

    # --- CATCH-ALL FOR THE "86 NULLS" ---
    'All others': 'Other / General',
    'Other': 'Other / General',
    'other': 'Other / General',
    'Charlie': 'Other / General',
    'BL-LA': 'Other / General',
    'Student & Entry Level': 'Entry Level'
}

# Implementation
final_df['category'] = final_df['category'].map(simple_mapping)

In [ ]:
field_mapping = {
    # --- TECH & DIGITAL ---
    'Software Engineering': 'Software Engineering',
    'Web Development': 'Software Engineering',
    'Blockchain & Crypto': 'Software Engineering',

    'AI / Machine Learning': 'AI & Data Science',
    'Data Science & Analytics': 'AI & Data Science',

    'DevOps & IT Infrastructure': 'DevOps, Cloud & Security',
    'Information Security': 'DevOps, Cloud & Security',

    # --- SALES & REVENUE ---
    'Sales': 'B2B & Enterprise Sales',
    'Business Development': 'B2B & Enterprise Sales',
    'SaaS Strategy': 'B2B & Enterprise Sales',
    'Customer Success & Support': 'Customer Success & Care',
    'Retail & Hospitality': 'Customer Success & Care',

    # --- MARKETING & CONTENT ---
    'Marketing': 'Digital Marketing',
    'Content & Writing': 'Content & Creative',
    'Creative & Design': 'Content & Creative',

    # --- CORPORATE OPS & MANAGEMENT ---
    'Product Management': 'Business Ops & Project Management',
    'Project Management': 'Business Ops & Project Management',
    'Management & Leadership': 'Business Ops & Project Management',
    'Operations': 'Business Ops & Project Management',
    'Consulting': 'Business Ops & Project Management',
    'Administrative Support': 'Business Ops & Project Management',

    'Finance & Accounting': 'Finance, Tax & Payroll',
    'Payroll Services': 'Finance, Tax & Payroll',
    'Legal & Compliance': 'Finance, Tax & Payroll',

    'HR & Recruiting': 'HR & Talent Management',
    'Training & Education': 'HR & Talent Management',

    # --- INDUSTRY SPECIFIC ---
    'Healthcare & Clinical': 'Healthcare & Life Sciences',
    'Specialized Engineering': 'Industrial, Safety & EHS',
    'Safety & Environment': 'Industrial, Safety & EHS',

    # --- MISCELLANEOUS ---
    'Other / General': 'Other / General',
    'Entry Level': 'Other / General'
}

# Implementation
final_df['fields'] = final_df['category'].map(field_mapping)

In [ ]:
final_df.head(3)

,job_id,title,company,location,category,tags,description,source,url,date_posted,scrape_date,tags_list,region,fields
0,4cf90830d8d06e1969b33c1b50c448fc,AI Trainer,Anuttacon,Worldwide,AI / Machine Learning,"video, ai/ml, research, game design, data anal...",🎮 Our Culture: Our culture thrives on creativi...,Remotive,https://remotive.com/remote-jobs/ai-ml/ai-trai...,2026-02-12T21:01:01,2026-02-15,"[video, ai/ml, research, game design, data ana...",Global/Remote,AI & Data Science
1,19df2af9c128af93c6c7cc91673367a6,Office Assistant,Coalition Technologies,Worldwide,Marketing,"css, excel, frontend, git, html, illustrator, ...",WHY YOU SHOULD APPLY: Coalition Technologies i...,Remotive,https://remotive.com/remote-jobs/marketing/off...,2026-02-11T20:16:31,2026-02-15,"[css, excel, frontend, git, html, illustrator,...",Global/Remote,Digital Marketing
2,03d4570f5ac4f4c5df59981ceb653da5,iOS Developer,nooro,USA,Software Engineering,"api, backend, git, ios, security, swift, ui/ux...","WHO ARE WE? At nooro, we're revolutionizing pa...",Remotive,https://remotive.com/remote-jobs/software-deve...,2026-02-09T21:15:55,2026-02-15,"[api, backend, git, ios, security, swift, ui/u...",North America,Software Engineering


In [ ]:

import ast
final_df['tags_list'] = final_df['tags_list'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_skills = final_df.explode('tags_list')
df_skills = df_skills.rename(columns={'tags_list': 'skill'})

df_skills['skill'] = df_skills['skill'].str.lower()

In [ ]:
# List of words that are 'noise' for a skill analysis
noise_words = ['support', 'management', 'senior', 'technical', 'assistant', 'manager', 'director', "remote"]

# Filtering exploded skills dataframe
df_skills = df_skills[~df_skills['skill'].isin(noise_words)]

# Now look at the top 10
print(df_skills['skill'].value_counts().head(10))

skill
sales            34
software         30
engineering      29
growth           28
security         25
engineer         24
operational      24
lead             24
digital nomad    22
health           21
Name: count, dtype: int64


In [ ]:
unify_map = {
    "software engineering": ["software", "engineering", "engineer", "developer", "coding"],
    "Data Science": [ "data science"],
    "Machine Learning": ["ml", "machine learning"],
    'Artificial Intelligence': ["ai", "artificial intelligence"],
    "cybersecurity": ["security", "infosec", "cyber", "compliance"],
}

def unify_skills(skill):
    skill = str(skill).lower().strip()
    for clean_name, synonyms in unify_map.items():
        if any(syn in skill for syn in synonyms):
            return clean_name
    return skill

# Apply to exploded dataframe
df_skills['skill_clean'] = df_skills['skill'].apply(unify_skills)

In [ ]:
# Apply to exploded dataframe
df_skills['skill_clean'].value_counts().head(30)

,count
skill_clean,
software engineering,204
cybersecurity,49
sales,34
growth,28
lead,24
operational,24
Artificial Intelligence,23
digital nomad,22
Machine Learning,22


In [ ]:
final_df.isnull().sum()

,0
job_id,0
title,0
company,0
location,0
category,95
tags,0
description,0
source,0
url,0
date_posted,0


##**ANALYSIING DATA**

In [ ]:
source_counts = final_df['source'].value_counts().reset_index()
source_counts.columns = ['source', 'Count']

fig = px.pie(
    source_counts,
    values='Count',
    names='source',
    title='Source Distribution',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# 3. Clean up the labels
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(showlegend=True)

fig.show()

In [ ]:
region_counts = final_df['region'].value_counts().reset_index()
region_counts.columns = ['region', 'Count']

fig = px.pie(
    region_counts,
    values='Count',
    names='region',
    title='Region Distribution',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# 3. Clean up the labels
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(showlegend=True)

fig.show()

Observation: Since the Data was obtained from remote website therefore half of the portion is Global and remote

In [ ]:
final_df['fields']

,fields
0,AI & Data Science
1,Digital Marketing
2,Software Engineering
3,B2B & Enterprise Sales
4,Software Engineering
...,...
335,Other / General
336,Other / General
337,Other / General
338,Digital Marketing


In [ ]:
fields_counts = final_df['fields'].value_counts().reset_index()
fields_counts.columns = ['fields', 'Count']

fig = px.pie(
    fields_counts,
    values='Count',
    names='fields',
    title='Fields Distribution',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# 3. Clean up the labels
fig.update_traces(textposition='inside', textinfo='percent')
fig.update_layout(showlegend=True)

fig.show()

Observation: The software Engineering occupies quarter of the job market

In [ ]:
se = final_df[final_df['fields'] == "Software Engineering"]

se_counts = se['category'].value_counts().reset_index()
se_counts.columns = ['category', 'Count']

fig = px.pie(
    se_counts,
    values='Count',
    names='category',
    title='Job Distribution (Software Engineering)',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# 3. Clean up the labels
fig.update_traces(textposition='inside', textinfo='percent')
fig.update_layout(showlegend=True)

fig.show()

Observation: The software development leads the software engineering field where as there are some jobs available for web devs as well.

In [ ]:
b = final_df[final_df['fields'] == "Business Ops & Project Management"]

b_counts = b['category'].value_counts().reset_index()
b_counts.columns = ['category', 'Count']

fig = px.pie(
    b_counts,
    values='Count',
    names='category',
    title='Job Distribution (Business Ops & Project Management)',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)

# 3. Clean up the labels
fig.update_traces(textposition='inside', textinfo='percent')
fig.update_layout(showlegend=True)

fig.show()

Observation: The core management jobs leads in busniness ops & project management where as 61% market is captured by these two fields alone

In [ ]:
overall_top_skills = df_skills['skill_clean'].value_counts().head(10)
print("Top 10 Global Skills:\n", overall_top_skills)

fig = px.bar(overall_top_skills, title="Top 10 Most Demanding Skills Overall",
             labels={'value': 'Frequency', 'index': 'Skill'}, color=overall_top_skills.values)
fig.show()

Top 10 Global Skills:
 skill_clean
software engineering       204
cybersecurity               49
sales                       34
growth                      28
lead                        24
operational                 24
Artificial Intelligence     23
digital nomad               22
Machine Learning            22
health                      21
Name: count, dtype: int64


In [ ]:
# export file
final_df.to_csv("jobs_dataset_plus.csv", index=False)